# Solana Data Management

In [ ]:
import asyncio, time
from typing import Dict, Any, Optional, Tuple
from decimal import Decimal, getcontext, ROUND_DOWN
from datetime import datetime
from cachetools import TTLCache
from cachetools_async import cached
from logging_system import setup_logging, AppLogger
from pumpfun import PumpFunSubscriptions
from copy_trading.data_management.solana_manager import SolanaTxAnalyzer
from copy_trading.position_management.models import TraderTradeData

setup_logging(console_output=True, file_output=False, min_level_to_process='DEBUG')
logger = AppLogger(name="SolanaDataManagement")
getcontext().prec = 26
cache: TTLCache[str, Any] = TTLCache(maxsize=100, ttl=10)

solana_analyzer = SolanaTxAnalyzer()
await solana_analyzer.start()


In [ ]:
def create_trade_data_from_pumpfun(data: Dict[str, Any]) -> Optional[TraderTradeData]:
    try:
        return TraderTradeData(
            # Información básica del trade
            trader_wallet=data.get('traderPublicKey', ''),
            side=data.get('txType', '').lower(),
            token_address=data.get('mint', ''),
            amount_sol=format(Decimal(str(data.get('solAmount', 0))), "f"),
            signature=data.get('signature', ''),

            # Información del token
            token_amount=format(Decimal(str(data.get('tokenAmount', 0))), "f"),
            new_token_balance=format(Decimal(str(data.get('newTokenBalance', 0))), "f"),

            # Información del pool/bonding curve
            pool=data.get('pool', ''),
            bonding_curve_key=data.get('bondingCurveKey', ''),
            v_tokens_in_bonding_curve=format(Decimal(str(data.get('vTokensInBondingCurve', 0))), "f"),
            v_sol_in_bonding_curve=format(Decimal(str(data.get('vSolInBondingCurve', 0))), "f"),
            market_cap_sol=format(Decimal(str(data.get('marketCapSol', 0))), "f"),

            # Metadatos
            timestamp=datetime.now()
        )
    except Exception as e:
        logger.error(f"Error creating TradeData from PumpFun: {e}")

In [ ]:
@cached(cache=cache, key=lambda wallet: wallet)
async def get_sol_balance(wallet: str) -> str:
    try:
        result = await solana_analyzer.get_sol_balance(wallet)
        return result
    except Exception as e:
        logger.error(f"Error getting SOL balance: {e}")
        return "0.0"

In [ ]:
# (trader_wallet, token_address) -> monto original en SOL del primer trade de apertura
first_trade: Dict[Tuple[str, str], Decimal] = {}

async def calculate_percentage_of_token_in_balance(data: Dict[str, Any]) -> str:
    global first_trade
    try:
        start_time = time.time()

        trade_data = create_trade_data_from_pumpfun(data)
        if trade_data is None:
            logger.warning("Could not create TradeData from PumpFun")
            return "0.0"

        balance = Decimal(await get_sol_balance(trade_data.trader_wallet))
        sol_amount = Decimal(trade_data.amount_sol)

        base_key = (trade_data.trader_wallet, trade_data.token_address)
        base_first_amount = first_trade.get(base_key, Decimal(0))

        percentage = Decimal("0.0")
        if base_first_amount == 0 and trade_data.side == "buy":
            first_trade[base_key] = sol_amount

            token_amount = Decimal(trade_data.token_amount)
            new_token_balance = Decimal(trade_data.new_token_balance)
            price_sol_per_token = first_trade[base_key] / token_amount
            token_balance_in_sol = new_token_balance * price_sol_per_token

            if token_balance_in_sol == 0 or balance == 0:
                logger.warning(f"Cannot calculate percentage of token in balance, token_balance_in_sol: {token_balance_in_sol}, balance: {balance}")
                return "0.0"

            percentage = (token_balance_in_sol / (balance + token_balance_in_sol)) * 100
        else:
            if sol_amount == 0 or balance == 0:
                logger.warning(f"Cannot calculate percentage of token in balance, sol_amount: {sol_amount}, balance: {balance}")
                return "0.0"

            if trade_data.side == "buy":
                percentage = (sol_amount / (balance + sol_amount)) * 100
            else:
                percentage = (sol_amount / balance) * 100

        # Update cache
        # El balance siempre será un Future, así que debemos esperar el resultado, operar y volver a guardar como Future
        balance_future = cache[trade_data.trader_wallet]
        balance_cached = await balance_future
        balance_cached = Decimal(str(balance_cached))
        if trade_data.side == "buy":
            new_balance = balance_cached - sol_amount
        else:
            new_balance = balance_cached + sol_amount

        loop = asyncio.get_event_loop()
        cache[trade_data.trader_wallet] = loop.create_future()
        cache[trade_data.trader_wallet].set_result(str(new_balance))

        # Information
        logger.info("="*100)
        side_emoji = "🟢" if trade_data.side == "buy" else "🔴"
        logger.info(f"{side_emoji} Side: {trade_data.side}")
        logger.info(f"👛 Owner: {trade_data.trader_wallet}")
        logger.info(f"🪙 Mint: {trade_data.token_address}")
        logger.info(f"💰 General Balance: {balance} SOL")
        if trade_data.side == "buy":
            logger.info(f"🚀 Sol sent: -{trade_data.amount_sol} SOL")
            logger.info(f"🔢 Tokens received: +{trade_data.token_amount}")
            logger.info(f"📊 Percentage sent: -{percentage.quantize(Decimal('0.000001'), rounding=ROUND_DOWN).normalize()}%")
        else:
            logger.info(f"🚀 Sol received: +{trade_data.amount_sol} SOL")
            logger.info(f"🔢 Tokens sent: -{trade_data.token_amount}")
            logger.info(f"📊 Percentage received: +{percentage.quantize(Decimal('0.000001'), rounding=ROUND_DOWN).normalize()}%")
        logger.info(f"⏱️ Execution time: {time.time() - start_time:.6f} seconds")
        logger.info("="*100)

        return f"{percentage:.2f}"
    except Exception as e:
        logger.error(f"Error calculating percentage of token in balance: {e}", exc_info=True)
        return "0.0"

In [ ]:
try:
    async with PumpFunSubscriptions() as pumpfun_subscriptions:
        await pumpfun_subscriptions.subscribe_account_trade(
            account_addresses=[
                "GMN2f6PsBwUKKpuxQdsJGveJKsNEgiH7APVwdfBtnAtz",
                "suqh5sHtr8HyJ7q8scBimULPkPpA557prMG47xCHQfK",
                "j1oAbxxiDUWvoHxEDhWE7THLjEkDQW2cSHYn2vttxTF",
                "j1opmdubY84LUeidrPCsSGskTCYmeJVzds1UWm6nngb",
                "DfMxre4cKmvogbLrPigxmibVTTQDuzjdXojWzjCXXhzj",
                "73LnJ7G9ffBDjEBGgJDdgvLUhD5APLonKrNiHsKDCw5B",
                "HdxkiXqeN6qpK2YbG51W23QSWj3Yygc1eEk2zwmKJExp",
                "FjmRj8y9xfDaj5Aygq88t5jAFbpxrbZ16JNPPG1sx9FQ",
                "JD1dHSqYkrXvqUVL8s6gzL1yB7kpYymsHfwsGxgwp55h",
                "ZG98FUCjb8mJ824Gbs6RsgVmr1FhXb2oNiJHa2dwmPd",
                "JD38n7ynKYcgPpF7k1BhXEeREu1KqptU93fVGy3S624k",
                "niggerd597QYedtvjQDVHZTCCGyJrwHNm2i49dkm5zS",
                "AtTjQKXo1CYTa2MuxPARtr382ZyhPU5YX4wMMpvaa1oy",
                "JDd3hy3gQn2V982mi1zqhNqUw1GfV2UL6g76STojCJPN",
                "nya666pQkP3PzWxi7JngU3rRMHuc7zbLK8c8wxQ4qpT",
                "4DdrfiDHpmx55i4SPssxVzS9ZaKLb8qr45NKY9Er9nNh",
                "2QMXjBufQprAfEutzh4gMWRuMp1impcjk5BLpfYEHAsb",
                "8KKii2e5YMQuDT32uuYzKUDjNaLrWbMjzu72cfcw6fac",
                "4hSXPtxZgXFpo6Vxq9yqxNjcBoqWN3VoaPJWonUtupzD",
                "2rbMgYvzAb3xDk6vXrzKkY3VwsmyDZsJTkvB3JJYsRzA",
                "9FNz4MjPUmnJqTf6yEDbL1D4SsHVh7uA8zRHhR5K138r",
                "MNhBbrscBPmeid54buiqSgyWa4D8PY6uKHoK2wJsTJN",
                "BwaVFCDJ4HfRfFWq1S23LHkk5VF4GKEw9oz7F1PxgcHv",
                "9ZzjXiwkGRDBwVHJitfx8AmnN2YUbnqW6M1tH38juEeJ",
                "3tc4BVAdzjr1JpeZu6NAjLHyp4kK3iic7TexMBYGJ4Xk"
            ],
            callback=calculate_percentage_of_token_in_balance
        )
        while True:
            await asyncio.sleep(1)
except asyncio.CancelledError:
    await solana_analyzer.stop()
    print("Canceled")

## Importaciones

In [ ]:
from copy_trading.data_management.solana_manager import SolanaTxAnalyzer
from copy_trading.data_management.solana_manager import SolanaWebsocketManager
import asyncio
from logging_system import setup_logging

setup_logging(console_output=True, file_output=False, min_level_to_process='DEBUG')

### Account Info

In [ ]:
async with SolanaTxAnalyzer() as solana_analizer:
    result = await solana_analizer.analyze_transaction_by_signature(
        signature="4tQxggo22JHXkX1DeV1rWbL8Uc8uDgovNSEdknpUMeAUH9qyuTPeUBWdkAttA7uwctrK5vwz7Lj6if7ZMkTFVVH"
    )

result

### Http - Signature for statuses

In [ ]:
async with SolanaTxAnalyzer() as solana_analizer:
    result = await solana_analizer.get_signatures_with_statuses(signatures=[
        "2XA5khyKPmYTwwH6vqAz6i1BjwxnQwEE5wzF83PoozwU1UkXD6vkearNrQrVrjb5ktYkky67trHZVgHY4zJop73f",
        "5RzBEYz9kTGz6Yp9xuou9kSAiV44NTsvNhf9jho2bQx66djsZyqeTn48RK3DsP1G3JmAQ49EZLJMhSuaSmMbhT13",
        "3fVeh7JcK6mAe9Eq4Ybi7fXNxKNBoMnTAj1up6NuhKAKK3MCYWzfYJTaAZ5iC33Q3tmaMsgCPZwmfRs2bNzsXDHA",
        "4mjyUGwqw9vX98Tc5GcBrFAotGfPi8vbNi8ZMm9DQPSeX8hCEBJctKoXWyndKq9NZ4quUz5HAvacEA4pv5SqtWuG",
        "65BZKHiq69XD2sSmc9Sqs6Fvh1kMeAKJ6gbRWtJ3H8cVrmj1e6gHyqkeiJcAToVzwjKEykh9WB2ZgVkQk9DnXTSr"
    ])

result

### Websocket

In [ ]:
async def on_notification(signature, msg):
    print(f"[{signature}] Notificación recibida:", msg)

async def on_timeout(signature, timeout):
    print(f"[{signature}] Timeout de {timeout} segundos.")

async def on_error(error: Exception):
    try:
        print(f"{error}")
        print(type(error))
    except Exception as err:
        print("Se ejecuto un error:", err)

In [ ]:
signature = "3vy6K436cb5z1jgfQ9wrvXJMD3vLjts276oRZhkSUQL7Zb8RCvQn3ymWr8SkjQ9ejVyoXXhRK5dwVZSzSFGarEAM"

In [ ]:
async with SolanaWebsocketManager() as solana_websocket:
    solana_websocket.set_callbacks(
        on_signature_confirmed=on_notification,
        on_signature_timeout=on_timeout,
        on_connection_error=on_error
    )

    await solana_websocket.subscribe_signature(signature)

    await asyncio.sleep(70)